In [1]:
from datasets import load_dataset
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.cuda as cuda

C:\everything\Projects\Python Projects\image_classification\my_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Setting up cuda
device = 'cuda' if cuda.is_available() else 'cpu'
print(f"The current device is: {device}")

The current device is: cuda


In [3]:
# Loading the dataset from Hugging Face
train_dataset = load_dataset("ylecun/mnist", split= 'train')
test_dataset = load_dataset("ylecun/mnist", split = 'test')

In [4]:
# Turn image into Torch tensor
train_dataset = train_dataset.with_format("torch")
test_dataset = test_dataset.with_format("torch")

In [24]:
# Collate function to turn dtype into torch expected dtype for forward() and loss
def collate_func(batch):
    image = torch.stack([item['image'].type(torch.float) for item in batch])
    label = torch.stack([item['label'].type(torch.uint8) for item in batch])
    return {'image':image, 'label':label}

In [25]:
# Turn data into Torch DataLoader object
train_dataloader = DataLoader(train_dataset,
                              batch_size=32,
                              shuffle=True,
                              collate_fn=collate_func)
test_dataloader = DataLoader(test_dataset,
                             batch_size=32,
                             shuffle=True,
                             collate_fn=collate_func)

In [26]:
# Building the model
class ImageClassificationModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.convolution_block_1 = nn.Sequential(
            nn.Conv2d(
                in_channels=1,
                out_channels=8,
                kernel_size=(3,3),
                padding=1,
                stride=1
            ),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=(3,3),
                        stride=3,
                        padding=1),
            nn.ReLU()
        )
        self.connected_block = nn.Sequential(
            nn.Flatten(),
            nn.Linear(in_features=8 * 10 * 10,
                      out_features=32),
            nn.ReLU(),
            nn.Linear(in_features=32,
                      out_features=10)
        )
    def forward(self, x : torch.Tensor):
        outputs = self.connected_block(self.convolution_block_1(x))
        return outputs

In [27]:
# Instantiate the model, optimizer and loss function
model = ImageClassificationModel()
optimizer = optim.Adam(model.parameters(), lr=0.01)
loss_function = nn.CrossEntropyLoss()

model.to(device)
loss_function.to(device)

CrossEntropyLoss()

In [28]:
# Create the train loop for the model
epochs = 3
for epoch in range(epochs):
    for batch in train_dataloader:
        # Putting batch of data into device to avoide overflowing
        image, label = batch['image'].to(device), batch['label'].to(device)
        # Passing forward
        outputs = model(image)
        loss = loss_function(outputs, label)